# A bounded agent loop from scratch

## Learning goals

- Implement the observe-decide-act cycle explicitly
- Validate model decisions as either tool actions or final answers
- Track steps, tool calls, repeated actions, and consecutive errors
- Produce safe traces and explicit stop reasons
- Recognize when a workflow is simpler than an agent


## What makes this an agent loop?

In a workflow, code decides the next step. In an agent loop, the model can choose between an allowed tool action and a final answer based on the current observations. The runtime still controls every boundary: available tools, authorization, execution, budgets, and stopping.

An agent is not “a model in a `while True` loop.” A useful agent runtime has typed decisions, bounded resources, explicit state, controlled errors, and a result that explains why execution stopped.


## Mental model

```mermaid
stateDiagram-v2
  [*] --> Decide
  Decide --> Final: final answer
  Decide --> Validate: proposed tool action
  Validate --> Stop: invalid or unauthorized beyond limit
  Validate --> Execute: allowed and valid
  Execute --> Observe: result or controlled error
  Observe --> Stop: budget, repetition, or error limit
  Observe --> Decide: observation added to context
  Final --> [*]
  Stop --> [*]
```

Every cycle changes explicit runtime state. The loop stops on success *or* on a named safety/reliability condition. “The model will know when to stop” is not a runtime policy.


## Model decisions form a discriminated union

The model must choose exactly one shape: a tool action or a final answer. A discriminator gives both the model and parser an unambiguous contract.


In [ ]:
from typing import Annotated, Any, Literal

from pydantic import BaseModel, ConfigDict, Field, TypeAdapter


class ToolAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    kind: Literal["tool"]
    name: str = Field(min_length=1)
    arguments: dict[str, Any]


class FinalAnswer(BaseModel):
    model_config = ConfigDict(extra="forbid")
    kind: Literal["final"]
    answer: str = Field(min_length=1, max_length=2_000)


Decision = Annotated[ToolAction | FinalAnswer, Field(discriminator="kind")]
DecisionAdapter = TypeAdapter(Decision)

parsed = DecisionAdapter.validate_json(
    '{"kind": "tool", "name": "get_weather", "arguments": {"city": "Jaipur"}}'
)
print(parsed)


## A minimal tool runtime

The preceding notebook built a full registry. This notebook uses one small offline tool so we can focus on loop mechanics. The same principle holds: the runtime validates arguments and returns a structured observation.


In [ ]:
from pydantic import ValidationError


class WeatherArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")
    city: str = Field(min_length=2, max_length=100)


class Observation(BaseModel):
    tool: str
    ok: bool
    result: Any | None = None
    error: str | None = None


def execute_action(action: ToolAction) -> Observation:
    if action.name != "get_weather":
        return Observation(tool=action.name, ok=False, error="unknown_tool")
    try:
        args = WeatherArgs.model_validate(action.arguments)
    except ValidationError:
        return Observation(tool=action.name, ok=False, error="invalid_arguments")
    return Observation(
        tool=action.name,
        ok=True,
        result={
            "city": args.city,
            "condition": "warm and dry (offline mock)",
            "source": "teaching stub",
        },
    )


## State and limits are first-class data

Different limits answer different questions. `max_steps` bounds model decisions; `max_tool_calls` bounds external actions; `max_consecutive_errors` prevents a broken plan from thrashing. Repetition detection catches identical actions even before those totals are exhausted.


In [ ]:
from dataclasses import dataclass, field


@dataclass(frozen=True)
class RuntimeLimits:
    max_steps: int = 6
    max_tool_calls: int = 4
    max_consecutive_errors: int = 2


@dataclass
class RuntimeState:
    step: int = 0
    tool_calls: int = 0
    consecutive_errors: int = 0
    seen_actions: set[str] = field(default_factory=set)
    observations: list[Observation] = field(default_factory=list)
    trace: list[dict[str, Any]] = field(default_factory=list)


class AgentResult(BaseModel):
    status: Literal[
        "completed",
        "max_steps",
        "max_tool_calls",
        "repeated_action",
        "too_many_errors",
    ]
    answer: str | None = None
    steps: int
    tool_calls: int
    trace: list[dict[str, Any]]


## Use a scripted model to test the loop

A deterministic decision source lets us test runtime behavior without API keys or model variability. In production, `next_decision` would call a model with the user request, tool schemas, and accumulated observations, then parse the result with `DecisionAdapter`.


In [ ]:
class ScriptedModel:
    def __init__(self, decisions: list[Decision]):
        self.decisions = iter(decisions)

    def next_decision(
        self,
        user_request: str,
        observations: list[Observation],
    ) -> Decision:
        try:
            return next(self.decisions)
        except StopIteration:
            return FinalAnswer(kind="final", answer="No scripted decision remains.")


## Implement the bounded loop

The trace records decisions and outcomes, not hidden chain-of-thought. Action signatures use canonical JSON so dictionary key order does not defeat repetition detection.


In [ ]:
import json


def action_signature(action: ToolAction) -> str:
    return json.dumps(
        {"name": action.name, "arguments": action.arguments},
        sort_keys=True,
        separators=(",", ":"),
    )


def stop_result(status: str, state: RuntimeState) -> AgentResult:
    return AgentResult(
        status=status,
        steps=state.step,
        tool_calls=state.tool_calls,
        trace=state.trace,
    )


def run_agent(
    model: ScriptedModel,
    user_request: str,
    limits: RuntimeLimits = RuntimeLimits(),
) -> AgentResult:
    state = RuntimeState()

    for step in range(1, limits.max_steps + 1):
        state.step = step
        decision = model.next_decision(user_request, state.observations)

        if isinstance(decision, FinalAnswer):
            state.trace.append({"step": step, "event": "final"})
            return AgentResult(
                status="completed",
                answer=decision.answer,
                steps=state.step,
                tool_calls=state.tool_calls,
                trace=state.trace,
            )

        if state.tool_calls >= limits.max_tool_calls:
            return stop_result("max_tool_calls", state)

        signature = action_signature(decision)
        if signature in state.seen_actions:
            state.trace.append({"step": step, "event": "repeated_action", "tool": decision.name})
            return stop_result("repeated_action", state)
        state.seen_actions.add(signature)

        observation = execute_action(decision)
        state.tool_calls += 1
        state.observations.append(observation)
        state.trace.append({
            "step": step,
            "event": "tool_result",
            "tool": decision.name,
            "ok": observation.ok,
        })

        state.consecutive_errors = 0 if observation.ok else state.consecutive_errors + 1
        if state.consecutive_errors >= limits.max_consecutive_errors:
            return stop_result("too_many_errors", state)

    return stop_result("max_steps", state)


## Test success and failure paths

A runtime test suite should cover more than the happy path. Here we verify completion, repeated-action stopping, and consecutive invalid-tool stopping.


In [ ]:
success_model = ScriptedModel([
    ToolAction(kind="tool", name="get_weather", arguments={"city": "Jaipur"}),
    FinalAnswer(kind="final", answer="The offline mock reports warm, dry weather."),
])
success = run_agent(success_model, "What is Jaipur weather?")
print(success.model_dump())

repeated = ToolAction(kind="tool", name="get_weather", arguments={"city": "Jaipur"})
repeat_result = run_agent(ScriptedModel([repeated, repeated]), "Check again forever")
print(repeat_result.model_dump())

error_result = run_agent(
    ScriptedModel([
        ToolAction(kind="tool", name="unknown_a", arguments={}),
        ToolAction(kind="tool", name="unknown_b", arguments={}),
    ]),
    "Use unavailable tools",
)
print(error_result.model_dump())

assert success.status == "completed"
assert repeat_result.status == "repeated_action"
assert error_result.status == "too_many_errors"


## Production budgets to add

- total input and output tokens;
- wall-clock deadline and per-tool timeout;
- monetary budget or provider quota;
- maximum observation size;
- maximum parallel actions;
- maximum consecutive model-format failures;
- explicit approval state for write actions.

Counters should be enforced by the runtime and included in safe telemetry. A prompt asking the model to conserve tokens is not a hard budget.


## Failure cases and improvements

| Failure | Runtime response |
|---|---|
| Repeats the same call | Canonical signature and repeated-action stop |
| Alternates two failing tools | Consecutive-error and total-step limits |
| Invents a tool | Unknown-tool observation; never dynamic import/eval |
| Produces malformed decision JSON | Parse failure counter and bounded retry |
| Returns an answer after budget expiry | Runtime rejects further steps |
| Tool succeeds but result is huge | Truncate or store externally before observation |
| Trace leaks reasoning or secrets | Record structured events and redacted outcomes only |


## Exercise

1. Add a wall-clock deadline to `run_agent`.
2. Add a maximum observation-character budget.
3. Detect a two-action cycle, not only identical consecutive actions.
4. Add a `needs_confirmation` result for a proposed write tool.
5. Describe a task from your domain that should remain a workflow instead of becoming an agent.


## Summary

A safe agent loop alternates typed model decisions with validated tool observations under explicit runtime limits. It stops on a final answer, budget exhaustion, repetition, or too many errors—and reports which condition occurred. The model chooses among capabilities; the runtime owns permissions, execution, accounting, and stopping.
